In [2]:
import os
import pandas as pd
import numpy as np

PROJECT_ROOT = r"C:\path\to\your\supply_chain_project"   # <-- EDIT THIS (same path as your other 3 scripts)
OUT = os.path.join(PROJECT_ROOT, "outputs")
os.makedirs(OUT, exist_ok=True)

df = pd.read_csv(f"{OUT}/simulation_results.csv", parse_dates=["date"])

RECOVERY_THRESHOLD = 99.0
DISRUPTION_START = 20

scenarios = [s for s in df["scenario"].unique() if s != "baseline"]
baseline_cost_total = df[df.scenario == "baseline"]["daily_cost"].sum()

kpi_rows = []
for scenario in scenarios:
    s = df[df.scenario == scenario].sort_values("day").reset_index(drop=True)

    overall_service_level = 100 * s["fulfilled"].sum() / s["demand"].sum()
    trough_service_level = s["service_level_%"].min()
    trough_day = s.loc[s["service_level_%"].idxmin(), "day"]

    # Recovery time: first day at/after disruption start where service level
    # returns to >= threshold AND stays there for the rest of the window.
    post_disruption = s[s["day"] >= DISRUPTION_START]
    recovered_mask = post_disruption["service_level_%"] >= RECOVERY_THRESHOLD
    # find first index from which all subsequent values are True
    recovery_day = None
    for idx in post_disruption.index:
        if recovered_mask.loc[idx:].all():
            recovery_day = post_disruption.loc[idx, "day"]
            break
    recovery_time_days = (recovery_day - DISRUPTION_START) if recovery_day is not None else np.nan

    scenario_cost_total = s["daily_cost"].sum()
    extra_cost = scenario_cost_total - baseline_cost_total
    extra_cost_pct = 100 * extra_cost / baseline_cost_total

    total_unfulfilled_units = s["unfulfilled"].sum()
    total_expedited_units = s["expedited_units"].sum()

    kpi_rows.append({
        "scenario": scenario,
        "overall_service_level_%": round(overall_service_level, 2),
        "trough_service_level_%": round(trough_service_level, 2),
        "trough_day": int(trough_day),
        "recovery_time_days": recovery_time_days,
        "total_cost_$": round(scenario_cost_total, 0),
        "extra_cost_vs_baseline_$": round(extra_cost, 0),
        "extra_cost_vs_baseline_%": round(extra_cost_pct, 1),
        "total_unfulfilled_units": round(total_unfulfilled_units, 0),
        "total_expedited_units": round(total_expedited_units, 0),
    })

kpi_df = pd.DataFrame(kpi_rows)

# ---- Composite Network Robustness Score (0-100, higher = more robust) ----
def normalize(col, invert=False):
    lo, hi = kpi_df[col].min(), kpi_df[col].max()
    if hi == lo:
        return pd.Series([1.0] * len(kpi_df))
    norm = (kpi_df[col] - lo) / (hi - lo)
    return (1 - norm) if invert else norm

service_score = normalize("trough_service_level_%")          # higher trough = better
recovery_score = normalize("recovery_time_days", invert=True)  # lower recovery time = better
cost_score = normalize("extra_cost_vs_baseline_%", invert=True)  # lower cost overrun = better

kpi_df["network_robustness_score"] = (
    100 * (0.4 * service_score + 0.3 * recovery_score + 0.3 * cost_score)
).round(1)

kpi_df.to_csv(f"{OUT}/kpi_summary.csv", index=False)

print("RESILIENCE KPI SUMMARY")
print("=" * 100)
print(kpi_df.to_string(index=False))
print("\nSaved: kpi_summary.csv")

print("\nPlain-English readout:")
for _, row in kpi_df.iterrows():
    print(f"\n  {row['scenario'].replace('_', ' ').title()}:")
    print(f"    - Fulfillment dropped to a low of {row['trough_service_level_%']}% "
          f"on day {row['trough_day']} of the simulation.")
    print(f"    - It took {row['recovery_time_days']:.0f} days after the disruption began "
          f"for service to return to normal (>= {RECOVERY_THRESHOLD}%).")
    print(f"    - This disruption cost an extra ${row['extra_cost_vs_baseline_$']:,.0f} "
          f"({row['extra_cost_vs_baseline_%']}% over baseline operating cost).")
    print(f"    - Network Robustness Score: {row['network_robustness_score']}/100")


RESILIENCE KPI SUMMARY
        scenario  overall_service_level_%  trough_service_level_%  trough_day  recovery_time_days  total_cost_$  extra_cost_vs_baseline_$  extra_cost_vs_baseline_%  total_unfulfilled_units  total_expedited_units  network_robustness_score
supplier_failure                    98.27                   72.88          28                  14      483404.0                   15939.0                       3.4                    986.0                 2395.0                     100.0
    port_closure                    95.78                   57.50          28                  14      492186.0                   24720.0                       5.3                   2408.0                 1516.0                      30.0

Saved: kpi_summary.csv

Plain-English readout:

  Supplier Failure:
    - Fulfillment dropped to a low of 72.88% on day 28 of the simulation.
    - It took 14 days after the disruption began for service to return to normal (>= 99.0%).
    - This disruption cost 